# core

> Claude code api backend for fastllm

In [ ]:
#| default_exp core

In [ ]:
#| export
import json, uuid, asyncio, re
from datetime import datetime, timezone
from pathlib import Path
from claude_agent_sdk import (query, ClaudeAgentOptions, create_sdk_mcp_server, tool,
    AssistantMessage, ToolUseBlock, StreamEvent, ResultMessage)
from fastllm.types import *
from fastllm.anthropic import (norm_sse_event, norm_tool_calls, norm_parts,
    norm_finish, norm_usage, finalize_usage, denorm_msgs, delta_index_fn)
from fastllm.streaming import mk_acollect_stream
from fastspec.errors import APIError

In [ ]:
#| export
MCP_SERVER_NAME = "fastllm"
WORK_DIR = Path.home() / ".fastllm-claude-agent"
WORK_DIR.mkdir(exist_ok=True)

def _proj_dir(cwd):
    san = re.sub(r'[^a-zA-Z0-9]', '-', str(Path(cwd).resolve()))
    return Path.home() / ".claude/projects" / san

In [ ]:
_proj_dir(WORK_DIR)

Path('/Users/keremturgutlu/.claude/projects/-Users-keremturgutlu--fastllm-claude-agent')

In [ ]:
#| export
SERVER_TOOLS = ["WebSearch", "WebFetch"]

def msgs_to_jsonl(msgs, model="claude-sonnet-4-6", session_id=None):
    "Convert fastllm Msgs to a CC session JSONL string."
    sid = session_id or str(uuid.uuid4())
    ts = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")
    records, prev_uuid = [], None
    for d in denorm_msgs(msgs):
        for b in d.get("content", []):
            if isinstance(b, dict) and b.get("type") == "tool_use":
                nm = b.get("name", "")
                if nm and not nm.startswith("mcp__") and nm not in SERVER_TOOLS: b["name"] = f"{MCP_PREFIX}{nm}"
        u = str(uuid.uuid4())
        r = {"type": "user" if d["role"] == "user" else "assistant",
             "sessionId": sid, "uuid": u, "parentUuid": prev_uuid,
             "isSidechain": False, "permissionMode": "default", "timestamp": ts,
             "message": {"type": "message", **d}}
        if d["role"] == "assistant": r["message"]["model"] = model
        records.append(r); prev_uuid = u
    return sid, '\n'.join(json.dumps(r) for r in records) + '\n'

In [ ]:
#| export
def mk_stub(name, desc, schema, block):
    @tool(name, desc, schema)
    async def _stub(args): await block.wait()
    return _stub

In [ ]:
#| export
def _last_user_text(m):
    "Extract text from a user Msg's content parts."
    return "\n".join(p.text or '' for p in m.content if p.type == PartType.text)

def claude_mk_payload(msgs, model, stream=False, **kwargs):
    "Build prompt + options for Claude Code SDK query. Last msg is the new turn; rest is resumed history."
    system, tools = kwargs.get('system'), kwargs.get('tools')
    if msgs and msgs[-1].role == 'user' and _last_user_text(msgs[-1]):
        *history, last = msgs
        prompt = _last_user_text(last)
    else:
        history, prompt = msgs, "." # continue tool results, works fine but if it becomes an issue make tool use prompt
    
    # if history:
    #     sid, jsonl = msgs_to_jsonl(history, model=model)
    #     print(f"=== PROMPT: {prompt!r} ===")
    #     print(jsonl)
    
    block = asyncio.Event()
    mcp_tools, allowed = [], []
    for t in (tools or []):
        nm, desc, params = fn_schema(t)
        if nm:
            mcp_tools.append(mk_stub(nm, desc or "", params, block))
            allowed.append(f"mcp__{MCP_SERVER_NAME}__{nm}")
    mcp_servers = {MCP_SERVER_NAME: create_sdk_mcp_server(MCP_SERVER_NAME, tools=mcp_tools)} if mcp_tools else {}
    cc_tools = ["WebSearch", "WebFetch"] if kwargs.get('web_search_options') is not None else []

    opt_kw = dict(model=model, env={'ANTHROPIC_API_KEY': ''}, cwd=str(WORK_DIR),
        include_partial_messages=True, permission_mode="bypassPermissions",
        system_prompt=system or "", mcp_servers=mcp_servers,
        allowed_tools=allowed, strict_mcp_config=True, tools=cc_tools)

    # opt_kw['stderr'] = lambda s: print(f"[CC stderr] {s}")

    log_path = WORK_DIR / "cc-logs" / f"{datetime.now():%Y%m%d-%H%M%S-%f}.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    opt_kw['stderr'] = lambda s, p=log_path: p.open('a').write(s + '\n')

    if history:
        sid, jsonl = msgs_to_jsonl(history, model=model)
        pd = _proj_dir(WORK_DIR); pd.mkdir(parents=True, exist_ok=True)
        (pd / f"{sid}.jsonl").write_text(jsonl)
        opt_kw['resume'] = sid
    opts = ClaudeAgentOptions(**opt_kw)
    return {"prompt": prompt, "options": opts, "block": block}

In [ ]:
#| export
MCP_PREFIX = f"mcp__{MCP_SERVER_NAME}__"

async def claude_acollect_stream(payload, **kwargs):
    opts, prompt = payload["options"], payload["prompt"]
    async def _gen():
        gen = query(prompt=prompt, options=opts)
        saw_tool = False
        base, max_in_msg = 0, -1          # global index offset across messages
        try:
            async for msg in gen:
                if isinstance(msg, ResultMessage) and msg.is_error:
                    txt = msg.result or "; ".join(msg.errors or []) or msg.subtype
                    m = re.search(r'\b(\d{3})\b', txt or '')
                    raise APIError(txt, provider='claude_code', model=opts.model,
                                status_code=int(m.group(1)) if m else None, raw=msg)
                if isinstance(msg, AssistantMessage):
                    if any(isinstance(b, ToolUseBlock) and b.name.startswith(MCP_PREFIX) for b in (msg.content or [])): saw_tool = True
                elif isinstance(msg, StreamEvent):
                    ev = msg.event
                    t = ev.get("type")
                    if t == "message_start":
                        base += max_in_msg + 1      # advance past prev message's blocks
                        max_in_msg = -1
                    elif t in ("content_block_start","content_block_delta","content_block_stop") and "index" in ev:
                        i = ev["index"]
                        max_in_msg = max(max_in_msg, i)
                        ev = {**ev, "index": i + base}
                    if t == "message_stop" and saw_tool: return
                    cb = ev.get("content_block", {})
                    if cb.get("type") == "tool_use" and cb.get("name", "").startswith(MCP_PREFIX):
                        ev = {**ev, "content_block": {**cb, "name": cb["name"][len(MCP_PREFIX):]}}
                    delta = norm_sse_event(ev)
                    for tc in (delta.tool_calls or []):
                        if tc.name in ["WebSearch", "WebFetch"]: tc.server = True
                    yield delta
        finally:
            try: await gen.aclose()
            except Exception: pass
    async for o in mk_acollect_stream(_gen(), index_fn=delta_index_fn, api_name='claude_code', **kwargs): yield o

In [ ]:
#| export
api_registry.register('claude_code',
    norm_tool_calls=norm_tool_calls, norm_parts=norm_parts,
    norm_finish=norm_finish, norm_usage=norm_usage, finalize_usage=finalize_usage,
    mk_payload=claude_mk_payload, acollect_stream=claude_acollect_stream,
    cost=lambda *_: 0)

### Tests

In [ ]:
from fastllm.chat import mk_msgs, acomplete, lite_mk_func, AsyncChat

In [ ]:
msgs = mk_msgs("What is 2+2?")
r = await acomplete(msgs, 'claude-sonnet-4-6', api_name='claude_code', stream=True)
async for o in r:
    if isinstance(o, Completion): print(f"\n--- finish: {o.finish_reason}, usage: {o.usage}")
    elif t := o.get('text'): print(t, end='')

2 + 2 = **4**
--- finish: stop, usage: Usage(prompt_tokens=14, completion_tokens=14, total_tokens=28, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 14, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 14})


In [ ]:
def simple_add(a: int, b: int) -> int:
    "Add two numbers"
    return a + b

In [ ]:
msgs = mk_msgs("What is 3+5 and 10_5? Use the simple_add tool in parallel.")
r = await acomplete(msgs, 'claude-sonnet-4-6', api_name='claude_code', stream=True, tools=[lite_mk_func(simple_add)])

async for o in r:
    if isinstance(o, Completion):
        print(f"\n--- finish: {o.finish_reason}")
        print(f"--- tool_calls: {o.tool_calls}")
    elif isinstance(o, Part): print(f"\n[Part {o.type}: {o.data}]")
    elif t := o.get('text'): print(t, end='')

Sure! Let me calculate both sums simultaneously using the `simple_add` tool in parallel!
[Part tool_use: {'caller': {'type': 'direct'}, 'id': 'toolu_016U2rnSpG3G2Lxwa4dvZ4ew', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False}]



[Part tool_use: {'caller': {'type': 'direct'}, 'id': 'toolu_01U4E8SdzCRU9JrUg8PiALiw', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 5}, 'server': False}]



--- finish: tool_calls


--- tool_calls: [ToolCall(id='toolu_016U2rnSpG3G2Lxwa4dvZ4ew', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'caller': {'type': 'direct'}}), ToolCall(id='toolu_01U4E8SdzCRU9JrUg8PiALiw', name='simple_add', arguments={'a': 10, 'b': 5}, server=False, extra={'caller': {'type': 'direct'}})]


In [ ]:
def delta_text(o):
    "Extract printable content from streaming delta, return None if nothing to print"
    if isinstance(o, Part) and o.type == PartType.tool_result: 
        return f'🔧 {o.data['name']}\n'
    if isinstance(o,dict): 
        if o.get('thinking'):    return '🧠'
        elif txt:=o.get('text'): return txt
    return None

In [ ]:
chat = AsyncChat('claude-sonnet-4-6', api_name='claude_code', tools=[simple_add])
res = await chat("What is 7+3? Use the tool.", stream=True)
async for o in res: print(delta_text(o) or '', end='')

🔧 simple_add


The result of **7 + 3 = 10**. The tool confirmed this calculation successfully!

In [ ]:
def multiply(a: int, b: int) -> int:
    "Multiply two numbers"
    return a * b

chat = AsyncChat('claude-sonnet-4-6', api_name='claude_code', tools=[simple_add, multiply])
res = await chat("Calculate 3+5 and 4*6 in parallel using tools.", stream=True, max_steps=5)
async for o in res: print(delta_text(o) or '', end='')

I'll calculate both operations simultaneously by calling both tools in parallel!🔧 simple_add


🔧 multiply


Here are the results of both calculations, performed in parallel:

| Operation | Result |
|-----------|--------|
| 3 + 5     | **8**  |
| 4 × 6     | **24** |

Both tools were called simultaneously, making the process efficient! 🚀

In [ ]:
res = await chat("What was the last result.", stream=True, max_steps=5)
async for o in res: print(delta_text(o) or '', end='')

The last results were:

- **3 + 5 = 8** (Addition)
- **4 × 6 = 24** (Multiplication)

These were the two calculations performed in parallel in our previous interaction.

In [ ]:
chat = AsyncChat('claude-sonnet-4-6', api_name='claude_code', search='l')
res = await chat("Can you search the web for weather in Istanbul", stream=True)
async for o in res: print(delta_text(o) or '', end='')

🔧 web_search


Here is the current weather in **Istanbul, Türkiye** for today, **Wednesday, June 24, 2026**:

☀️ **Current Conditions:**
- [*](https://www.accuweather.com/en/tr/istanbul/318251/weather-forecast/318251 "Istanbul, Istanbul, Türkiye Weather Forecast | AccuWeather")**Temperature:** 80°F with sunny skies
- [*](https://www.accuweather.com/en/tr/istanbul/318251/weather-forecast/318251 "Istanbul, Istanbul, Türkiye Weather Forecast | AccuWeather")**RealFeel®:** 85°F | **Heat Index:** 80°F
- [*](https://www.accuweather.com/en/tr/istanbul/318251/weather-forecast/318251 "Istanbul, Istanbul, Türkiye Weather Forecast | AccuWeather")**Wind:** NE at 12 mph, with gusts up to 14 mph
- [*](https://www.accuweather.com/en/tr/istanbul/318251/weather-forecast/318251 "Istanbul, Istanbul, Türkiye Weather Forecast | AccuWeather")**Air Quality:** Fair

🌡️ **Today's Forecast:**
- [*](https://www.accuweather.com/en/tr/istanbul/318251/weather-forecast/318251 "Istanbul, Istanbul, Türkiye Weather Forecast | AccuWeat

[*](https://www.accuweather.com/en/tr/istanbul/318251/weather-forecast/318251 "Istanbul, Istanbul, Türkiye Weather Forecast | AccuWeather")Temperatures will climb through the afternoon, peaking at 85°F around 3 PM, then gradually cooling to 80°F by 7 PM.

Overall, it's a warm and sunny summer day in Istanbul! 🌞

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()